
### Topics Covered:
1. **Linear Regression** — Fitting a line to data
2. **Bayesian Classifier** — 1 feature, 2 classes
3. **Naive Bayes Classifier** — 2 features, 2 classes

---

In [ ]:
# ── Setup: install/import everything we need ──────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.stats import norm
from sklearn.datasets import make_classification

np.random.seed(42)
plt.rcParams.update({'figure.dpi': 110, 'font.size': 12})
print("✅ All imports successful!")

---
# Part 1 — Linear Regression

##  What problem does it solve?
Given a set of input–output pairs **(x, y)**, find the straight line that **best fits** the data, so we can **predict y for new x values**.

---

##  The Model

We assume a linear relationship:

$$\hat{y} = w_0 + w_1 x$$

| Symbol | Name | Meaning |
|--------|------|---------|
| $\hat{y}$ | Predicted output | What our model thinks y is |
| $w_0$ | Bias / intercept | Where the line crosses the y-axis |
| $w_1$ | Weight / slope | How steeply the line rises |
| $x$ | Feature | Our input variable |

---

##  The Loss: Mean Squared Error (MSE)

We measure how **wrong** our line is using:

$$\mathcal{L}(w_0, w_1) = \frac{1}{N} \sum_{i=1}^{N} (y_i - \hat{y}_i)^2$$

- Each $(y_i - \hat{y}_i)$ is the **residual** (vertical gap between a point and the line).
- We **square** them so positives and negatives don't cancel.
- We want to **minimize** this loss.

---

##  Closed-Form Solution (Ordinary Least Squares)

Taking the derivative of MSE and setting it to zero gives us:

$$w_1 = \frac{\sum_{i=1}^{N}(x_i - \bar{x})(y_i - \bar{y})}{\sum_{i=1}^{N}(x_i - \bar{x})^2}$$

$$w_0 = \bar{y} - w_1 \bar{x}$$

Where $\bar{x}$ and $\bar{y}$ are the **means** of x and y.



In [ ]:
# ── 1.1 Generate some toy data ────────────────────────────────────────────────
# Imagine: x = hours studied, y = exam score
N = 30
x = np.linspace(1, 10, N)
true_w0, true_w1 = 20, 7          # True intercept and slope
noise = np.random.normal(0, 5, N)  # Gaussian noise
y = true_w0 + true_w1 * x + noise

print(f"Dataset: {N} samples")
print(f"x range: [{x.min():.1f}, {x.max():.1f}]")
print(f"y range: [{y.min():.1f}, {y.max():.1f}]")
print(f"True relationship: y = {true_w0} + {true_w1}*x + noise")

In [ ]:
# ── 1.2 Compute weights using the OLS closed-form formulas ────────────────────
x_mean = np.mean(x)
y_mean = np.mean(y)

# Numerator: sum of (xi - x_bar)(yi - y_bar)
numerator   = np.sum((x - x_mean) * (y - y_mean))
# Denominator: sum of (xi - x_bar)^2
denominator = np.sum((x - x_mean) ** 2)

w1 = numerator / denominator
w0 = y_mean - w1 * x_mean

print("=== Learned Parameters ===")
print(f"  w0 (intercept) = {w0:.3f}   (true: {true_w0})")
print(f"  w1 (slope)     = {w1:.3f}   (true: {true_w1})")

In [ ]:
# ── 1.3 Predict and compute MSE ───────────────────────────────────────────────
y_pred = w0 + w1 * x
residuals = y - y_pred
mse = np.mean(residuals ** 2)

print(f"MSE on training data: {mse:.3f}")
print(f"RMSE (same units as y): {np.sqrt(mse):.3f}")

# Predict for a new value
x_new = 8.0
y_new = w0 + w1 * x_new
print(f"\nPrediction: If hours studied = {x_new}, predicted score = {y_new:.1f}")

In [ ]:
# ── 1.4 Visualize: Data + Fitted Line + Residuals ─────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Plot 1: Data + Line ---
ax = axes[0]
ax.scatter(x, y, color='steelblue', zorder=5, label='Data points', s=60)
ax.plot(x, y_pred, color='tomato', linewidth=2.5, label=f'Fitted line: ŷ = {w0:.1f} + {w1:.1f}x')

# Draw residuals as vertical lines
for xi, yi, ypi in zip(x, y, y_pred):
    ax.plot([xi, xi], [yi, ypi], color='gray', linewidth=0.8, alpha=0.6)

ax.scatter([x_new], [y_new], color='gold', s=150, zorder=6, marker='*', label=f'New prediction (x={x_new})')
ax.set_xlabel('Hours Studied (x)')
ax.set_ylabel('Exam Score (y)')
ax.set_title('Linear Regression: Data + Fitted Line')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# --- Plot 2: Residuals ---
ax2 = axes[1]
colors = ['tomato' if r < 0 else 'steelblue' for r in residuals]
ax2.bar(range(N), residuals, color=colors, alpha=0.8, width=0.6)
ax2.axhline(0, color='black', linewidth=1.5)
ax2.set_xlabel('Sample Index')
ax2.set_ylabel('Residual (y - ŷ)')
ax2.set_title('Residuals (Random = Good Fit!)')
ax2.grid(True, alpha=0.3, axis='y')

blue_p = mpatches.Patch(color='steelblue', label='Positive residual')
red_p  = mpatches.Patch(color='tomato',    label='Negative residual')
ax2.legend(handles=[blue_p, red_p], fontsize=9)

plt.tight_layout()
plt.savefig('linear_regression.png', bbox_inches='tight')
plt.show()
print("✅ If residuals look random (no pattern), the linear model is a good fit!")

### Exercise — Linear Regression

**Q1.** If the slope $w_1 = 0$, what does that mean about the relationship between x and y?

**Q2.** If you see a **U-shaped** pattern in the residuals plot, is linear regression appropriate? Why?

**Q3.** MSE penalizes large errors more than small ones. Can you think of a situation where this is **undesirable**?


---
# Part 2 — Bayesian Classifier (1 Feature, 2 Classes)

##  What problem does it solve?
Given a **single measurement x**, decide which of **2 classes** (e.g., Healthy vs. Sick) it belongs to.

---

##  Bayes' Theorem — The Core Idea

$$P(C_k \mid x) = \frac{P(x \mid C_k) \; P(C_k)}{P(x)}$$

| Term | Name | Plain English |
|------|------|---------------|
| $P(C_k \mid x)$ | **Posterior** | Probability of class $k$ *given* we observed $x$ |
| $P(x \mid C_k)$ | **Likelihood** | How probable is this $x$ if we're in class $k$? |
| $P(C_k)$ | **Prior** | How common is class $k$ before seeing any data? |
| $P(x)$ | **Evidence** | Overall probability of observing $x$ (normalizer) |

---

##  Gaussian Likelihood

We model each class's feature distribution as a **Gaussian (Normal)**:

$$P(x \mid C_k) = \frac{1}{\sqrt{2\pi\sigma_k^2}} \exp\!\left(-\frac{(x - \mu_k)^2}{2\sigma_k^2}\right)$$

- $\mu_k$ = mean of feature $x$ in class $k$
- $\sigma_k^2$ = variance of feature $x$ in class $k$
- We **estimate** these from training data.

---

## Decision Rule

Assign $x$ to the class with the **highest posterior**:

$$\hat{C} = \arg\max_{k} \; P(C_k \mid x) = \arg\max_{k} \; P(x \mid C_k) \cdot P(C_k)$$

> We drop $P(x)$ since it's the same for both classes — it's just a normalizer!

The **decision boundary** is where $P(C_1 \mid x) = P(C_2 \mid x)$.

---

##  Example Scenario
- **Feature x** = Blood glucose level (mg/dL)
- **Class 0** = Healthy → $\mu_0 = 90$, $\sigma_0 = 10$
- **Class 1** = Diabetic → $\mu_1 = 140$, $\sigma_1 = 20$
- **Priors**: 70% healthy, 30% diabetic → $P(C_0)=0.7$, $P(C_1)=0.3$

In [ ]:
# ── 2.1 Define class parameters ───────────────────────────────────────────────
# Class 0: Healthy
mu0, sigma0 = 90, 10
prior0 = 0.7

# Class 1: Diabetic
mu1, sigma1 = 140, 20
prior1 = 0.3

print("Class Parameters:")
print(f"  Class 0 (Healthy):  μ={mu0}, σ={sigma0}, P(C0)={prior0}")
print(f"  Class 1 (Diabetic): μ={mu1}, σ={sigma1}, P(C1)={prior1}")

In [ ]:
# ── 2.2 Bayesian classification for a single point ────────────────────────────
def gaussian_pdf(x, mu, sigma):
    """P(x | C_k) — Gaussian likelihood"""
    return norm.pdf(x, loc=mu, scale=sigma)

def bayesian_classify(x_val, mu0, sigma0, prior0, mu1, sigma1, prior1):
    """Returns predicted class and posterior probabilities."""
    # Likelihoods
    likelihood0 = gaussian_pdf(x_val, mu0, sigma0)
    likelihood1 = gaussian_pdf(x_val, mu1, sigma1)
    
    # Unnormalized posteriors
    unnorm0 = likelihood0 * prior0
    unnorm1 = likelihood1 * prior1
    
    # Normalize (divide by evidence P(x))
    evidence = unnorm0 + unnorm1
    posterior0 = unnorm0 / evidence
    posterior1 = unnorm1 / evidence
    
    pred_class = 0 if posterior0 > posterior1 else 1
    return pred_class, posterior0, posterior1, likelihood0, likelihood1

# Test with a specific blood glucose value
x_test = 115  # mg/dL — in between the two means!
pred, p0, p1, l0, l1 = bayesian_classify(x_test, mu0, sigma0, prior0, mu1, sigma1, prior1)

print(f"=== Classification for x = {x_test} mg/dL ===")
print(f"  P(x | Healthy)  = {l0:.6f}")
print(f"  P(x | Diabetic) = {l1:.6f}")
print(f"  P(Healthy  | x) = {p0:.4f}  ({p0*100:.1f}%)")
print(f"  P(Diabetic | x) = {p1:.4f}  ({p1*100:.1f}%)")
print(f"  → Predicted Class: {'Healthy' if pred == 0 else 'Diabetic'} (Class {pred})")

In [ ]:
# ── 2.3 Find the decision boundary ───────────────────────────────────────────
# Decision boundary: where posterior0 = posterior1
# i.e., likelihood0 * prior0 = likelihood1 * prior1

x_range = np.linspace(40, 210, 1000)
posteriors0 = []
posteriors1 = []

for xv in x_range:
    _, p0_v, p1_v, _, _ = bayesian_classify(xv, mu0, sigma0, prior0, mu1, sigma1, prior1)
    posteriors0.append(p0_v)
    posteriors1.append(p1_v)

posteriors0 = np.array(posteriors0)
posteriors1 = np.array(posteriors1)

# Boundary = where classes flip
diff = posteriors0 - posteriors1
boundary_idx = np.argmin(np.abs(diff))
boundary_x = x_range[boundary_idx]
print(f"Decision boundary ≈ x = {boundary_x:.2f} mg/dL")
print(f"  x < {boundary_x:.1f} → Classify as Healthy")
print(f"  x > {boundary_x:.1f} → Classify as Diabetic")

In [ ]:
# ── 2.4 Visualization ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# --- Plot 1: Likelihoods ---
ax = axes[0]
pdf0 = gaussian_pdf(x_range, mu0, sigma0)
pdf1 = gaussian_pdf(x_range, mu1, sigma1)

ax.plot(x_range, pdf0 * prior0, 'steelblue', lw=2.5, label=f'P(x|Healthy)·P(C0)  [μ={mu0}, σ={sigma0}, prior={prior0}]')
ax.plot(x_range, pdf1 * prior1, 'tomato',    lw=2.5, label=f'P(x|Diabetic)·P(C1) [μ={mu1}, σ={sigma1}, prior={prior1}]')
ax.axvline(boundary_x, color='purple', linestyle='--', lw=2, label=f'Decision boundary ≈ {boundary_x:.1f}')
ax.axvline(x_test, color='gold', linestyle=':', lw=2, label=f'Test point x={x_test}')

ax.fill_between(x_range, pdf0 * prior0, alpha=0.15, color='steelblue')
ax.fill_between(x_range, pdf1 * prior1, alpha=0.15, color='tomato')
ax.set_xlabel('Blood Glucose Level (mg/dL)')
ax.set_ylabel('Likelihood × Prior')
ax.set_title('Gaussian Likelihoods (Scaled by Prior)')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# --- Plot 2: Posterior Probabilities ---
ax2 = axes[1]
ax2.plot(x_range, posteriors0, 'steelblue', lw=2.5, label='P(Healthy | x)')
ax2.plot(x_range, posteriors1, 'tomato',    lw=2.5, label='P(Diabetic | x)')
ax2.axvline(boundary_x, color='purple', linestyle='--', lw=2, label=f'Boundary ≈ {boundary_x:.1f}')
ax2.axvline(x_test, color='gold', linestyle=':', lw=2, label=f'Test x={x_test}')
ax2.axhline(0.5, color='gray', linestyle=':', lw=1)

ax2.scatter([x_test], [p0], color='steelblue', s=100, zorder=5)
ax2.scatter([x_test], [p1], color='tomato',    s=100, zorder=5)
ax2.annotate(f'P(H|x)={p0:.2f}', xy=(x_test, p0), xytext=(x_test+8, p0+0.05), fontsize=9)
ax2.annotate(f'P(D|x)={p1:.2f}', xy=(x_test, p1), xytext=(x_test+8, p1-0.08), fontsize=9)

ax2.set_xlabel('Blood Glucose Level (mg/dL)')
ax2.set_ylabel('Posterior Probability')
ax2.set_title('Posterior Probabilities')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)
ax2.set_ylim(-0.05, 1.05)

plt.tight_layout()
plt.savefig('bayesian_classifier.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── 2.5 Effect of Priors — Interactive Demo ───────────────────────────────────
print("=== How Prior Shifts the Decision Boundary ===")
print(f"{'Prior P(C1)':>15} | {'Boundary x':>12} | {'Classification at x=115':>25}")
print("-" * 58)

for p1_val in [0.1, 0.2, 0.3, 0.5, 0.7, 0.9]:
    p0_val = 1 - p1_val
    # Find boundary
    diffs = []
    for xv in x_range:
        _, pp0, pp1, _, _ = bayesian_classify(xv, mu0, sigma0, p0_val, mu1, sigma1, p1_val)
        diffs.append(pp0 - pp1)
    bx = x_range[np.argmin(np.abs(np.array(diffs)))]
    
    pred115, _, _, _, _ = bayesian_classify(115, mu0, sigma0, p0_val, mu1, sigma1, p1_val)
    label = 'Healthy' if pred115 == 0 else 'Diabetic'
    print(f"{p1_val:>15.1f} | {bx:>12.2f} | {label:>25}")

print("\n💡 As P(Diabetic) increases, boundary shifts LEFT → classifier is more aggressive about labeling Diabetic")

###  Exercise — Bayesian Classifier

**Q1.** If $P(C_0) = P(C_1) = 0.5$ (equal priors), how does the decision rule simplify?

**Q2.** Class 0 has $\sigma=10$ (narrow) and Class 1 has $\sigma=20$ (wide). For a very extreme value of x (say x=200), which class becomes more likely and why?

**Q3.** In a disease-screening scenario, you want to minimize **false negatives** (missing sick people). Should you increase or decrease $P(C_1)$ (the disease prior)? What happens to false positives?


---
# Part 3 — Naïve Bayes Classifier (2 Features, 2 Classes)

##  What's new here?
Now we have **2 features** $(x_1, x_2)$ instead of one. We extend the Bayesian framework — with one key **assumption**.

---

##  Full Joint Posterior

With 2 features, Bayes' theorem becomes:

$$P(C_k \mid x_1, x_2) \propto P(x_1, x_2 \mid C_k) \cdot P(C_k)$$

The joint likelihood $P(x_1, x_2 \mid C_k)$ is the hard part — it requires modeling feature correlations.

---

##  The Naïve Assumption: Conditional Independence

**Assumption:** Given the class $C_k$, features $x_1$ and $x_2$ are **independent**:

$$P(x_1, x_2 \mid C_k) = P(x_1 \mid C_k) \cdot P(x_2 \mid C_k)$$

This is **"naïve"** because features are rarely truly independent in the real world. But it works surprisingly well!

---

##  Full Classification Rule

$$\hat{C} = \arg\max_{k} \; P(C_k) \cdot \prod_{j=1}^{d} P(x_j \mid C_k)$$

For 2 features:

$$\hat{C} = \arg\max_{k} \; P(C_k) \cdot P(x_1 \mid C_k) \cdot P(x_2 \mid C_k)$$

---

##  Example Scenario
**Task:** Classify emails as Spam / Not Spam.
- **Feature $x_1$** = Number of exclamation marks in email
- **Feature $x_2$** = Email length in words
- **Class 0** = Not Spam → exclamation: $\mu=1, \sigma=1$; length: $\mu=200, \sigma=50$
- **Class 1** = Spam → exclamation: $\mu=8, \sigma=3$; length: $\mu=50, \sigma=20$
- **Priors**: 60% not spam, 40% spam

In [ ]:
# ── 3.1 Define 2-class, 2-feature Naïve Bayes parameters ─────────────────────
# Class 0: Not Spam
params = {
    0: {
        'prior': 0.6,
        'x1': {'mu': 1,   'sigma': 1},   # Exclamation marks
        'x2': {'mu': 200, 'sigma': 50},  # Email length (words)
        'label': 'Not Spam'
    },
    1: {
        'prior': 0.4,
        'x1': {'mu': 8,  'sigma': 3},    # Exclamation marks
        'x2': {'mu': 50, 'sigma': 20},   # Email length (words)
        'label': 'Spam'
    }
}

print("=== Model Parameters ===")
for k, p in params.items():
    print(f"\nClass {k}: {p['label']} (prior={p['prior']})")
    print(f"  x1 (exclamations): μ={p['x1']['mu']}, σ={p['x1']['sigma']}")
    print(f"  x2 (length):       μ={p['x2']['mu']}, σ={p['x2']['sigma']}")

In [ ]:
# ── 3.2 Naïve Bayes classifier function ──────────────────────────────────────
def naive_bayes_2class_2feature(x1_val, x2_val, params, verbose=True):
    """
    Classify (x1, x2) using Naïve Bayes.
    Returns: predicted class, posteriors dict
    """
    scores = {}  # Unnormalized posteriors
    likelihoods = {}
    
    for k, p in params.items():
        l1 = gaussian_pdf(x1_val, p['x1']['mu'], p['x1']['sigma'])
        l2 = gaussian_pdf(x2_val, p['x2']['mu'], p['x2']['sigma'])
        # Naïve Bayes: multiply likelihoods (independence assumption!)
        scores[k] = p['prior'] * l1 * l2
        likelihoods[k] = (l1, l2)
    
    # Normalize to get true posteriors
    total = sum(scores.values())
    posteriors = {k: s / total for k, s in scores.items()}
    pred_class = max(posteriors, key=posteriors.get)
    
    if verbose:
        print(f"=== Classifying: x1={x1_val} exclamations, x2={x2_val} words ===")
        for k, p in params.items():
            l1, l2 = likelihoods[k]
            print(f"  Class {k} ({p['label']}):")
            print(f"    P(x1|C{k}) = {l1:.6f}")
            print(f"    P(x2|C{k}) = {l2:.8f}")
            print(f"    Prior      = {p['prior']}")
            print(f"    Score      = {scores[k]:.2e}")
            print(f"    Posterior  = {posteriors[k]:.4f} ({posteriors[k]*100:.1f}%)")
        print(f"  → Predicted: {params[pred_class]['label']} (Class {pred_class})")
    
    return pred_class, posteriors

# Test case 1: Suspicious email
pred, post = naive_bayes_2class_2feature(x1_val=6, x2_val=40, params=params)

In [ ]:
# Test case 2: Normal email
print()
pred2, post2 = naive_bayes_2class_2feature(x1_val=2, x2_val=180, params=params)

In [ ]:
# ── 3.3 Decision Boundary in 2D Feature Space ─────────────────────────────────
x1_range = np.linspace(-1, 20, 200)
x2_range = np.linspace(0, 350, 200)
X1, X2 = np.meshgrid(x1_range, x2_range)

# Classify every point in the grid
Z = np.zeros(X1.shape)
P_spam = np.zeros(X1.shape)  # Store P(Spam | x1, x2)

for i in range(X1.shape[0]):
    for j in range(X1.shape[1]):
        pred_ij, post_ij = naive_bayes_2class_2feature(
            X1[i,j], X2[i,j], params, verbose=False)
        Z[i,j] = pred_ij
        P_spam[i,j] = post_ij[1]

print("Grid computed! Plotting decision boundary...")

In [ ]:
# ── 3.4 Visualization ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Generate synthetic training data for plotting
n_per_class = 80
x1_c0 = np.random.normal(params[0]['x1']['mu'], params[0]['x1']['sigma'], n_per_class)
x2_c0 = np.random.normal(params[0]['x2']['mu'], params[0]['x2']['sigma'], n_per_class)
x1_c1 = np.random.normal(params[1]['x1']['mu'], params[1]['x1']['sigma'], n_per_class)
x2_c1 = np.random.normal(params[1]['x2']['mu'], params[1]['x2']['sigma'], n_per_class)

# --- Plot 1: Decision Boundary ---
ax = axes[0]
ax.contourf(X1, X2, Z, levels=[-0.5, 0.5, 1.5],
            colors=['lightsteelblue', 'lightsalmon'], alpha=0.5)
ax.contour(X1, X2, Z, levels=[0.5], colors='purple', linewidths=2.5)

ax.scatter(x1_c0, x2_c0, c='steelblue', s=30, alpha=0.7, label='Not Spam (C0)', edgecolors='white', lw=0.3)
ax.scatter(x1_c1, x2_c1, c='tomato',    s=30, alpha=0.7, label='Spam (C1)',     edgecolors='white', lw=0.3)

# Plot our two test points
ax.scatter([6], [40],  c='black', s=200, marker='*', zorder=10, label='Test: (6 !!, 40 words) → Spam')
ax.scatter([2], [180], c='gold',  s=200, marker='*', zorder=10, label='Test: (2 !!, 180 words) → Ham')

ax.set_xlabel('x₁: Number of Exclamation Marks')
ax.set_ylabel('x₂: Email Length (words)')
ax.set_title('Naïve Bayes: Decision Boundary in 2D')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
ax.set_xlim(-1, 20)
ax.set_ylim(0, 350)

# Add region labels
ax.text(14, 280, 'Not Spam\nRegion', ha='center', fontsize=11,
        color='steelblue', fontweight='bold')
ax.text(14, 30, 'Spam\nRegion', ha='center', fontsize=11,
        color='tomato', fontweight='bold')

# --- Plot 2: P(Spam) heatmap ---
ax2 = axes[1]
im = ax2.contourf(X1, X2, P_spam, levels=50, cmap='RdBu_r')
ax2.contour(X1, X2, Z, levels=[0.5], colors='white', linewidths=2, linestyles='--')
plt.colorbar(im, ax=ax2, label='P(Spam | x₁, x₂)')

ax2.scatter(x1_c0, x2_c0, c='white', s=15, alpha=0.5, edgecolors='gray', lw=0.3)
ax2.scatter(x1_c1, x2_c1, c='black', s=15, alpha=0.5, edgecolors='gray', lw=0.3)

ax2.set_xlabel('x₁: Number of Exclamation Marks')
ax2.set_ylabel('x₂: Email Length (words)')
ax2.set_title('P(Spam | x₁, x₂) — Confidence Heatmap')
ax2.set_xlim(-1, 20)
ax2.set_ylim(0, 350)
ax2.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig('naive_bayes.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── 3.5 Marginal Distributions — Visualize Each Feature Separately ────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
x1_plot = np.linspace(-2, 20, 500)
x2_plot = np.linspace(0, 400, 500)

colors = ['steelblue', 'tomato']
labels = ['Not Spam (C0)', 'Spam (C1)']

# x1 distributions
for k in [0, 1]:
    p = params[k]
    pdf = gaussian_pdf(x1_plot, p['x1']['mu'], p['x1']['sigma'])
    axes[0].plot(x1_plot, pdf, color=colors[k], lw=2.5, label=labels[k])
    axes[0].fill_between(x1_plot, pdf, alpha=0.15, color=colors[k])
    axes[0].axvline(p['x1']['mu'], color=colors[k], lw=1.5, linestyle='--', alpha=0.7)

axes[0].set_xlabel('x₁: Number of Exclamation Marks')
axes[0].set_ylabel('P(x₁ | Class)')
axes[0].set_title('Feature 1 — Marginal Distributions')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# x2 distributions
for k in [0, 1]:
    p = params[k]
    pdf = gaussian_pdf(x2_plot, p['x2']['mu'], p['x2']['sigma'])
    axes[1].plot(x2_plot, pdf, color=colors[k], lw=2.5, label=labels[k])
    axes[1].fill_between(x2_plot, pdf, alpha=0.15, color=colors[k])
    axes[1].axvline(p['x2']['mu'], color=colors[k], lw=1.5, linestyle='--', alpha=0.7)

axes[1].set_xlabel('x₂: Email Length (words)')
axes[1].set_ylabel('P(x₂ | Class)')
axes[1].set_title('Feature 2 — Marginal Distributions')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Naïve Bayes: Each Feature is Modeled Independently Per Class', y=1.02, fontsize=12)
plt.tight_layout()
plt.savefig('nb_marginals.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── 3.6 Batch Classification + Accuracy on Synthetic Test Set ─────────────────
n_test = 200
# Generate test data from the true distributions
x1_test_c0 = np.random.normal(params[0]['x1']['mu'], params[0]['x1']['sigma'], n_test//2)
x2_test_c0 = np.random.normal(params[0]['x2']['mu'], params[0]['x2']['sigma'], n_test//2)
x1_test_c1 = np.random.normal(params[1]['x1']['mu'], params[1]['x1']['sigma'], n_test//2)
x2_test_c1 = np.random.normal(params[1]['x2']['mu'], params[1]['x2']['sigma'], n_test//2)

X1_test = np.concatenate([x1_test_c0, x1_test_c1])
X2_test = np.concatenate([x2_test_c0, x2_test_c1])
y_true  = np.array([0]*(n_test//2) + [1]*(n_test//2))

y_preds = []
for xi1, xi2 in zip(X1_test, X2_test):
    pred_i, _ = naive_bayes_2class_2feature(xi1, xi2, params, verbose=False)
    y_preds.append(pred_i)

y_preds = np.array(y_preds)
accuracy = np.mean(y_preds == y_true)

# Confusion matrix
TP = np.sum((y_preds == 1) & (y_true == 1))
TN = np.sum((y_preds == 0) & (y_true == 0))
FP = np.sum((y_preds == 1) & (y_true == 0))
FN = np.sum((y_preds == 0) & (y_true == 1))

print(f"=== Evaluation on {n_test} Test Samples ===")
print(f"  Accuracy: {accuracy*100:.1f}%")
print(f"\n  Confusion Matrix:")
print(f"  {'':15} | Predicted Ham | Predicted Spam")
print(f"  {'-'*50}")
print(f"  {'Actual Ham':15} | {TN:^13} | {FP:^14}")
print(f"  {'Actual Spam':15} | {FN:^13} | {TP:^14}")
print(f"\n  Precision (Spam): {TP/(TP+FP):.3f}")
print(f"  Recall    (Spam): {TP/(TP+FN):.3f}")

### Exercise — Naïve Bayes



**Q1.** What happens to $P(x_1 \mid C_k) \cdot P(x_2 \mid C_k)$ if one of the likelihoods is near zero? (Hint: think about what happens when you multiply many small numbers — this is the **zero probability / underflow** problem.)

**Q2.** How would you handle discrete features (e.g., does the email contain the word "FREE"?) in Naïve Bayes?
